In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

In [8]:
# English display and paths
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

SAVE_DIR = r'D:\Local\Mix\NTI-ML\Credit-card-Project\Visulization'
import os
os.makedirs(SAVE_DIR, exist_ok=True)

In [9]:
# ======================== 1. Data Loading and Initial Exploration ========================
print("="*60)
print("1. Data Loading and Initial Exploration")
print("="*60)

df = pd.read_csv(r'D:\Local\Mix\NTI-ML\Credit-card-Project\data\CC GENERAL.csv')
print(f"Dataset shape: {df.shape}")
print(f"Column names: {df.columns.tolist()}")
print(f"\nBasic dataset information:")
print(df.info())
print(f"\nStatistical description:")
print(df.describe().round(2))

# Missing value check
print(f"\nMissing value statistics:")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing count': missing, 'Missing percentage (%)': missing_pct})
print(missing_df[missing_df['Missing count'] > 0])

1. Data Loading and Initial Exploration
Dataset shape: (8950, 18)
Column names: ['CUST_ID', 'BALANCE', 'BALANCE_FREQUENCY', 'PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE', 'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY', 'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'CASH_ADVANCE_TRX', 'PURCHASES_TRX', 'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS', 'PRC_FULL_PAYMENT', 'TENURE']

Basic dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8950 entries, 0 to 8949
Data columns (total 18 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   CUST_ID                           8950 non-null   object 
 1   BALANCE                           8950 non-null   float64
 2   BALANCE_FREQUENCY                 8950 non-null   float64
 3   PURCHASES                         8950 non-null   float64
 4   ONEOFF_PURCHASES                  8950 non-null   flo

In [10]:
# ======================== 2. Data Preprocessing ========================
print("\n" + "="*60)
print("2. Data Preprocessing")
print("="*60)

# Copy data
df_clean = df.copy()

# 2.1 Handle missing values
print(f"MINIMUM_PAYMENTS Missing count: {df_clean['MINIMUM_PAYMENTS'].isnull().sum()}")
df_clean['MINIMUM_PAYMENTS'].fillna(df_clean['MINIMUM_PAYMENTS'].median(), inplace=True)
print(f"CREDIT_LIMIT Missing count: {df_clean['CREDIT_LIMIT'].isnull().sum()}")
df_clean['CREDIT_LIMIT'].fillna(df_clean['CREDIT_LIMIT'].median(), inplace=True)
print("Missing values filled successfully (using median imputation)")

# 2.2 Drop the CUST_ID column (not used for modeling)
cust_ids = df_clean['CUST_ID']
df_clean.drop('CUST_ID', axis=1, inplace=True)

# 2.3 Outlier handling — detected using the IQR method
print(f"\nOutlier detection (IQR method):")
outlier_summary = {}
for col in df_clean.columns:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_count = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    outlier_pct = round(outlier_count / len(df_clean) * 100, 2)
    outlier_summary[col] = {'Lower bound': round(lower,2), 'Upper bound': round(upper,2), 'Outlier count': outlier_count, 'Percentage %': outlier_pct}

outlier_df = pd.DataFrame(outlier_summary).T
print(outlier_df[outlier_df['Outlier count'] > 0].to_string())

# Winsorize outliers
for col in df_clean.columns:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_clean[col] = df_clean[col].clip(lower, upper)

print(f"\nOutlier winsorization completed")
print(f"Data shape after preprocessing: {df_clean.shape}")


2. Data Preprocessing
MINIMUM_PAYMENTS Missing count: 313
CREDIT_LIMIT Missing count: 1
Missing values filled successfully (using median imputation)

Outlier detection (IQR method):
                            Lower bound  Upper bound  Outlier count  Percentage %
BALANCE                        -2760.51      4942.93          695.0          7.77
BALANCE_FREQUENCY                  0.72         1.17         1493.0         16.68
PURCHASES                      -1566.11      2715.87          808.0          9.03
ONEOFF_PURCHASES                -866.11      1443.51         1013.0         11.32
INSTALLMENTS_PURCHASES          -702.96      1171.59          867.0          9.69
CASH_ADVANCE                   -1670.73      2784.55         1030.0         11.51
ONEOFF_PURCHASES_FREQUENCY        -0.45         0.75          782.0          8.74
CASH_ADVANCE_FREQUENCY            -0.33         0.56          525.0          5.87
CASH_ADVANCE_TRX                  -6.00        10.00          804.0          8.

In [11]:
# ======================== 3. Exploratory Data Analysis (EDA) ========================
print("\n" + "="*60)
print("3. Exploratory Data Analysis (EDA)")
print("="*60)

# 3.1 Feature distribution histograms
fig, axes = plt.subplots(6, 3, figsize=(16, 22))
axes = axes.flatten()
for i, col in enumerate(df_clean.columns):
    axes[i].hist(df_clean[col], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency')
# Hide unused subplots
for j in range(len(df_clean.columns), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature distribution histograms', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '01_feature_distribution_histograms.png'), dpi=150)
plt.close()
print("Figure 01: Feature distribution histograms saved")

# 3.2 Correlation heatmap
plt.figure(figsize=(16, 12))
corr_matrix = df_clean.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature correlation heatmap', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '02_correlation_heatmap.png'), dpi=150)
plt.close()
print("Figure 02: Correlation heatmap saved")

# 3.3 Boxplots of purchase behavior features
purchase_cols = ['PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'PURCHASES_FREQUENCY']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(purchase_cols):
    ax = axes[i//2, i%2]
    ax.boxplot(df_clean[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax.set_title(col, fontsize=12)
    ax.set_ylabel('Value')
plt.suptitle('Purchase behavior feature boxplots', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '03_purchase_behavior_boxplots.png'), dpi=150)
plt.close()
print("Figure 03: Purchase behavior boxplots saved")

# 3.4 Boxplots of cash advance features
cash_cols = ['CASH_ADVANCE', 'CASH_ADVANCE_FREQUENCY', 'CASH_ADVANCE_TRX']
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for i, col in enumerate(cash_cols):
    axes[i].boxplot(df_clean[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='lightcoral'))
    axes[i].set_title(col, fontsize=12)
plt.suptitle('Cash advance feature boxplots', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '04_cash_advance_boxplots.png'), dpi=150)
plt.close()
print("Figure 04: Cash advance boxplots saved")

# 3.5 PRC_FULL_PAYMENTDistribution (full payment ratio)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(df_clean['PRC_FULL_PAYMENT'], bins=50, color='green', edgecolor='white', alpha=0.8)
axes[0].set_title('Full payment ratio distribution')
axes[0].set_xlabel('PRC_FULL_PAYMENT')
axes[0].set_ylabel('Frequency')
axes[1].pie(
    [len(df_clean[df_clean['PRC_FULL_PAYMENT'] == 0]),
     len(df_clean[(df_clean['PRC_FULL_PAYMENT'] > 0) & (df_clean['PRC_FULL_PAYMENT'] < 1)]),
     len(df_clean[df_clean['PRC_FULL_PAYMENT'] == 1])],
    labels=['No full payment (0)', 'Partial payment (0~1)', 'Full payment (1)'],
    autopct='%1.1f%%', colors=['salmon', 'lightskyblue', 'lightgreen']
)
axes[1].set_title('Full payment category proportions')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '05_full_payment_distribution.png'), dpi=150)
plt.close()
print("Figure 05: Full payment distribution saved")

# 3.6 TENUREDistribution (credit card tenure)
plt.figure(figsize=(10, 5))
tenure_counts = df_clean['TENURE'].value_counts().sort_index()
plt.bar(tenure_counts.index, tenure_counts.values, color='teal', edgecolor='white')
plt.title('Credit card tenure(TENURE)Distribution', fontsize=14)
plt.xlabel('Tenure(months)')
plt.ylabel('Number of customers')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '06_tenure_distribution.png'), dpi=150)
plt.close()
print("Figure 06: Tenure distribution saved")

# 3.7 Balance vs. purchases scatter plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].scatter(df_clean['BALANCE'], df_clean['PURCHASES'], alpha=0.3, s=5, c='blue')
axes[0].set_xlabel('BALANCE'); axes[0].set_ylabel('PURCHASES')
axes[0].set_title('Balance vs. Purchases')
axes[1].scatter(df_clean['CREDIT_LIMIT'], df_clean['PURCHASES'], alpha=0.3, s=5, c='green')
axes[1].set_xlabel('CREDIT_LIMIT'); axes[1].set_ylabel('PURCHASES')
axes[1].set_title('Credit Limit vs. Purchases')
axes[2].scatter(df_clean['BALANCE'], df_clean['CASH_ADVANCE'], alpha=0.3, s=5, c='red')
axes[2].set_xlabel('BALANCE'); axes[2].set_ylabel('CASH_ADVANCE')
axes[2].set_title('Balance vs. Cash Advance')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '07_scatter_plot_matrix.png'), dpi=150)
plt.close()
print("Figure 07: Scatter plot matrix saved")



3. Exploratory Data Analysis (EDA)
Figure 01: Feature distribution histograms saved
Figure 02: Correlation heatmap saved
Figure 03: Purchase behavior boxplots saved
Figure 04: Cash advance boxplots saved
Figure 05: Full payment distribution saved
Figure 06: Tenure distribution saved
Figure 07: Scatter plot matrix saved


In [12]:
# ======================== 4. Feature Engineering ========================
print("\n" + "="*60)
print("4. Feature Engineering")
print("="*60)

# 4.1 Construct derived features
df_fe = df_clean.copy()

# Purchase-to-cash-advance ratio
df_fe['PURCHASE_RATIO'] = df_fe['PURCHASES'] / (df_fe['PURCHASES'] + df_fe['CASH_ADVANCE'] + 1)
# Credit utilization
df_fe['CREDIT_USAGE'] = df_fe['BALANCE'] / (df_fe['CREDIT_LIMIT'] + 1)
# Payment-to-balance ratio
df_fe['PAYMENT_TO_BALANCE'] = df_fe['PAYMENTS'] / (df_fe['BALANCE'] + 1)
# Average purchase amount per transaction
df_fe['AVG_PURCHASE_PER_TRX'] = df_fe['PURCHASES'] / (df_fe['PURCHASES_TRX'] + 1)
# Average cash advance per transaction
df_fe['AVG_CASH_PER_TRX'] = df_fe['CASH_ADVANCE'] / (df_fe['CASH_ADVANCE_TRX'] + 1)
# Installment purchase tendency
df_fe['INSTALLMENT_TENDENCY'] = df_fe['INSTALLMENTS_PURCHASES'] / (df_fe['PURCHASES'] + 1)

print("Derived features created. New features::")
new_features = ['PURCHASE_RATIO', 'CREDIT_USAGE', 'PAYMENT_TO_BALANCE',
                'AVG_PURCHASE_PER_TRX', 'AVG_CASH_PER_TRX', 'INSTALLMENT_TENDENCY']
print(new_features)


4. Feature Engineering
Derived features created. New features::
['PURCHASE_RATIO', 'CREDIT_USAGE', 'PAYMENT_TO_BALANCE', 'AVG_PURCHASE_PER_TRX', 'AVG_CASH_PER_TRX', 'INSTALLMENT_TENDENCY']


In [13]:
# ======================== 5. Data Standardization ========================
print("\n" + "="*60)
print("5. Data Standardization")
print("="*60)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_fe)
print(f"Shape after standardization: {X_scaled.shape}")
print(f"Mean: {X_scaled.mean():.6f}, Standard deviation: {X_scaled.std():.6f}")


5. Data Standardization
Shape after standardization: (8950, 23)
Mean: -0.000000, Standard deviation: 0.978019


In [14]:
# ======================== 6. PCA Dimensionality Reduction ========================
print("\n" + "="*60)
print("6. PCAPCA Dimensionality Reduction Analysis")
print("="*60)

pca = PCA(n_components=0.95)  # Retain 95% of variance
X_pca = pca.fit_transform(X_scaled)
print(f"Original number of features: {X_scaled.shape[1]}")
print(f"Number of principal components needed to retain 95% variance: {X_pca.shape[1]}")

# PCAExplained variance plot
pca_full = PCA()
pca_full.fit(X_scaled)
cumsum_var = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_)+1),
            pca_full.explained_variance_ratio_, color='steelblue')
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained variance ratio')
axes[0].set_title('Explained variance ratio by component')
axes[1].plot(range(1, len(cumsum_var)+1), cumsum_var, 'o-', color='darkred')
axes[1].axhline(y=0.95, color='gray', linestyle='--', label='95%Threshold')
axes[1].axhline(y=0.90, color='gray', linestyle=':', label='90%Threshold')
axes[1].set_xlabel('Number of principal components'); axes[1].set_ylabel('Cumulative explained variance ratio')
axes[1].set_title('Cumulative explained variance ratio')
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '08_PCAanalysis.png'), dpi=150)
plt.close()
print("Figure 08: PCAAnalysis saved")

# Use 2 principal components for visualization
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)
print(f"Explained variance ratio of the first 2 components: {pca_2d.explained_variance_ratio_.sum():.3f}")


6. PCAPCA Dimensionality Reduction Analysis
Original number of features: 23
Number of principal components needed to retain 95% variance: 13
Figure 08: PCAAnalysis saved
Explained variance ratio of the first 2 components: 0.496


In [15]:
# ======================== 7. K-Means Clustering ========================
print("\n" + "="*60)
print("7. K-Means Clustering Analysis")
print("="*60)

# 7.1 Use the Elbow Method + Silhouette Score to determine the optimal K
K_range = range(2, 11)
inertias = []
silhouette_scores = []
calinski_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_pca, labels))
    calinski_scores.append(calinski_harabasz_score(X_pca, labels))
    print(f"K={k}: Inertia={km.inertia_:.2f}, Silhouette={silhouette_scores[-1]:.4f}, CH={calinski_scores[-1]:.2f}")

# Plot K-value evaluation metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(list(K_range), inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].set_xlabel('K value'); axes[0].set_ylabel('Inertia (Within-cluster sum of squares)')
axes[0].set_title('Elbow Method — Determine the optimal K')
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.5, label='K=4 (Elbow point)')
axes[0].legend()

axes[1].plot(list(K_range), silhouette_scores, 's-', color='darkgreen', linewidth=2, markersize=8)
axes[1].set_xlabel('K value'); axes[1].set_ylabel('Silhouette Score (Silhouette Score)')
axes[1].set_title('Silhouette Score vs. K')
best_k_sil = list(K_range)[np.argmax(silhouette_scores)]
axes[1].axvline(x=best_k_sil, color='red', linestyle='--', alpha=0.5,
                label=f'Best K={best_k_sil}')
axes[1].legend()

axes[2].plot(list(K_range), calinski_scores, '^-', color='darkorange', linewidth=2, markersize=8)
axes[2].set_xlabel('K value'); axes[2].set_ylabel('Calinski-Harabasz Index')
axes[2].set_title('CHIndex vs. K')
best_k_ch = list(K_range)[np.argmax(calinski_scores)]
axes[2].axvline(x=best_k_ch, color='red', linestyle='--', alpha=0.5,
                label=f'Best K={best_k_ch}')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '09_K_value_evaluation.png'), dpi=150)
plt.close()
print("Figure 09: K-value evaluation plot saved")

# 7.2 Select the optimal K and build the final K-Means model
BEST_K = 4  # Based on the Elbow Method and Silhouette Score
print(f"\nSelected optimal K = {BEST_K}")

kmeans = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_pca)
df_fe['Cluster_KMeans'] = kmeans_labels

# K-Means Clustering visualization (PCA 2D) — Use the mean of each cluster in the 2D projection as the"centroid"
plt.figure(figsize=(10, 8))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#F7DC6F']
centroids_2d_list = []
for i in range(BEST_K):
    mask = kmeans_labels == i
    plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], c=colors[i],
                label=f'Cluster {i}', alpha=0.6, s=10)
    centroids_2d_list.append(X_pca_2d[mask].mean(axis=0))
centroids_2d = np.array(centroids_2d_list)
plt.scatter(centroids_2d[:, 0], centroids_2d[:, 1], c='black', marker='X',
            s=300, edgecolors='white', linewidth=2, label='Cluster centroids', zorder=5)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
plt.title(f'K-Means Clustering results (K={BEST_K})', fontsize=14)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '10_KMeansdbscan_clustering_visualization.png'), dpi=150)
plt.close()
print("Figure 10: K-MeansDBSCAN clustering visualization saved")


7. K-Means Clustering Analysis
K=2: Inertia=144866.44, Silhouette=0.2455, CH=2689.42
K=3: Inertia=121344.54, Silhouette=0.2401, CH=2472.35
K=4: Inertia=111768.47, Silhouette=0.2052, CH=2044.74
K=5: Inertia=103221.69, Silhouette=0.1905, CH=1845.52
K=6: Inertia=97267.56, Silhouette=0.1972, CH=1676.11
K=7: Inertia=91498.90, Silhouette=0.1859, CH=1578.63
K=8: Inertia=85513.21, Silhouette=0.1946, CH=1537.08
K=9: Inertia=81369.09, Silhouette=0.1978, CH=1470.20
K=10: Inertia=76100.26, Silhouette=0.1881, CH=1465.95
Figure 09: K-value evaluation plot saved

Selected optimal K = 4
Figure 10: K-MeansDBSCAN clustering visualization saved


In [17]:
# ======================== 10. Model ========================
print("\n" + "="*60)
print("10. Model Evaluation")
print("="*60)

evaluation_results = []

# K-Means Evaluation
km_sil = silhouette_score(X_pca, kmeans_labels)
km_db = davies_bouldin_score(X_pca, kmeans_labels)
km_ch = calinski_harabasz_score(X_pca, kmeans_labels)
evaluation_results.append(['K-Means', BEST_K, km_sil, km_db, km_ch])
print(f"K-Means: Silhouette={km_sil:.4f}, Davies-Bouldin={km_db:.4f}, CH={km_ch:.2f}")



10. Model Evaluation
K-Means: Silhouette=0.2052, Davies-Bouldin=1.6158, CH=2044.74


In [22]:
# ======================== 11. Model Evaluation Visualization ========================
print("\n" + "="*60)
print("11. Model Evaluation Visualization")
print("="*60)

eval_df = pd.DataFrame(evaluation_results, columns=['Model', 'Number of clusters', 'Silhouette Score', 'Davies-Bouldin Index', 'CHIndex'])

fig, axes = plt.subplots(1, 1, figsize=(14, 5))

# Silhouette Score comparison
methods = eval_df['Model'].tolist()
sil_vals = eval_df['Silhouette Score'].tolist()
bars = axes.bar(methods, sil_vals, color=['steelblue', 'darkgreen', 'darkorange'])
axes.set_ylabel('Silhouette Score')
axes.set_title('Model Silhouette Score comparison (higher is better)')
for bar, val in zip(bars, sil_vals):
    if not np.isnan(val):
        axes.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.4f}', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '15_model_evaluation.png'), dpi=150)
plt.close()
print("Figure 15: Model evaluation saved")



11. Model Evaluation Visualization
Figure 15: Model evaluation saved


In [23]:
# ======================== 12. Clustering Results Analysis — Customer Profiles ========================
print("\n" + "="*60)
print("12. Clustering Results Analysis — Customer Profiles")
print("="*60)

# Perform an in-depth analysis using the K-Means results (optimal)
df_fe_original = df_clean.copy()
df_fe_original['Cluster'] = kmeans_labels

# 12.1 Basic statistics for each cluster
cluster_stats = df_fe_original.groupby('Cluster').agg(['mean', 'std']).round(2)
print("\nMean feature values by cluster:")
print(df_fe_original.groupby('Cluster').mean().round(2))

# 12.2 Cluster sizes
cluster_sizes = df_fe_original['Cluster'].value_counts().sort_index()
print(f"\nNumber of samples per cluster:\n{cluster_sizes}")
print(f"Cluster proportions:\n{(cluster_sizes / len(df_fe_original) * 100).round(1)}%")

# 12.3 Cluster size pie chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].pie(cluster_sizes.values, labels=[f'Cluster {i}\n({v} customers)' for i, v in zip(cluster_sizes.index, cluster_sizes.values)],
            autopct='%1.1f%%', colors=colors, explode=(0.02, 0.02, 0.02, 0.02))
axes[0].set_title('Customer proportion by cluster', fontsize=14)

# 12.4 Key feature radar chart
key_features = ['BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'CREDIT_LIMIT',
                'PAYMENTS', 'PURCHASES_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'PRC_FULL_PAYMENT']
cluster_means = df_fe_original.groupby('Cluster')[key_features].mean()
# Normalize to 0–1 for the radar chart
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

angles = np.linspace(0, 2 * np.pi, len(key_features), endpoint=False).tolist()
angles += angles[:1]  # Close the radar chart

ax = axes[1]
ax = plt.subplot(1, 2, 2, projection='polar')
for i in range(BEST_K):
    values = cluster_means_norm.iloc[i].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {i}', color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(key_features, fontsize=8)
ax.set_title('Key feature radar chart by cluster', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '16_cluster_distribution_and_radar.png'), dpi=150)
plt.close()
print("Figure 16: Cluster distribution and radar chart saved")

# 12.5 Bar chart comparing key features across clusters
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()
for i, feature in enumerate(key_features):
    cluster_feature_means = df_fe_original.groupby('Cluster')[feature].mean()
    axes[i].bar(cluster_feature_means.index, cluster_feature_means.values, color=colors)
    axes[i].set_title(feature, fontsize=11)
    axes[i].set_xlabel('Cluster')
    axes[i].set_ylabel('Mean')
plt.suptitle('Mean key feature values across clusters', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '17_cluster_feature_comparison_bar_chart.png'), dpi=150)
plt.close()
print("Figure 17: Cluster feature comparison bar chart saved")

# 12.6 Feature heatmap across clusters
plt.figure(figsize=(14, 6))
cluster_profile = df_fe_original.groupby('Cluster')[key_features].mean()
sns.heatmap(cluster_profile.T, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Heatmap of mean key feature values by cluster', fontsize=14)
plt.xlabel('Cluster')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '18_cluster_feature_heatmap.png'), dpi=150)
plt.close()
print("Figure 18: Cluster feature heatmap saved")

# 12.7 TENURE distribution comparison across clusters
plt.figure(figsize=(14, 6))
for i in range(BEST_K):
    subset = df_fe_original[df_fe_original['Cluster'] == i]['TENURE']
    plt.hist(subset, bins=12, alpha=0.5, label=f'Cluster {i}', color=colors[i])
plt.title('Credit card tenure by cluster(TENURE)distribution comparison', fontsize=14)
plt.xlabel('Tenure(months)')
plt.ylabel('Number of customers')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '19_tenure_distribution_by_cluster.png'), dpi=150)
plt.close()
print("Figure 19: Tenure distribution by cluster saved")



12. Clustering Results Analysis — Customer Profiles

Mean feature values by cluster:
         BALANCE  BALANCE_FREQUENCY  PURCHASES  ONEOFF_PURCHASES  \
Cluster                                                            
0        1256.72               0.93      77.68             56.37   
1        1657.72               0.98    2162.92           1078.38   
2         468.44               0.89     507.24            203.66   
3        3746.97               0.98     516.56            272.74   

         INSTALLMENTS_PURCHASES  CASH_ADVANCE  PURCHASES_FREQUENCY  \
Cluster                                                              
0                         21.40       1029.67                 0.09   
1                        722.51        280.79                 0.90   
2                        287.01         31.88                 0.59   
3                        193.98       2345.61                 0.35   

         ONEOFF_PURCHASES_FREQUENCY  PURCHASES_INSTALLMENTS_FREQUENCY  \
Cluster    

In [26]:
# ======================== 13. Save Clustering Results ========================
print("\n" + "="*60)
print("13. Save Clustering Results")
print("="*60)

# Generate customer segment labels
df_result = df[['CUST_ID']].copy()
df_result['Cluster'] = kmeans_labels

# Cluster name mapping
cluster_names = {
    0: 'Low-activity customers',
    1: 'Frequent-purchase customers',
    2: 'Cash-advance customers',
    3: 'Steady-spending customers'
}

# Automatically name clusters based on actual data characteristics
cluster_avg = df_fe_original.groupby('Cluster')[['PURCHASES', 'CASH_ADVANCE', 'BALANCE', 'CREDIT_LIMIT']].mean()
print("Mean core metrics by cluster:")
print(cluster_avg)

# Save results
df_result['Cluster_Name'] = df_result['Cluster'].map(cluster_names)
df_result.to_csv(r'D:\Local\Mix\NTI-ML\Credit-card-Project\model\clustering_result.csv', index=False)
print("Clustering results saved to clustering_result.csv")

# Save the complete labeled dataset
df_fe_original.to_csv(r'D:\Local\Mix\NTI-ML\Credit-card-Project\model\data_with_clusters.csv', index=False)
print("Complete data with cluster labels saved to data_with_clusters.csv")


13. Save Clustering Results
Mean core metrics by cluster:
           PURCHASES  CASH_ADVANCE      BALANCE  CREDIT_LIMIT
Cluster                                                      
0          77.680880   1029.674343  1256.717504   2755.086531
1        2162.917110    280.791411  1657.716051   6432.586640
2         507.242003     31.879699   468.440796   3392.083709
3         516.561668   2345.605776  3746.971718   7232.269989
Clustering results saved to clustering_result.csv
Complete data with cluster labels saved to data_with_clusters.csv


In [27]:
# ======================== 14. Business Recommendations Summary ========================
print("\n" + "="*60)
print("14. Business Recommendations Summary")
print("="*60)

for cluster_id in sorted(cluster_names.keys()):
    size = cluster_sizes[cluster_id]
    pct = size / len(df_fe_original) * 100
    print(f"\n{'='*40}")
    print(f"Cluster {cluster_id}: {cluster_names[cluster_id]} (Customer count: {size}, Proportion: {pct:.1f}%)")
    print(f"{'='*40}")
    subset = df_fe_original[df_fe_original['Cluster'] == cluster_id]
    print(f"  Average balance: {subset['BALANCE'].mean():.2f}")
    print(f"  Average purchase amount: {subset['PURCHASES'].mean():.2f}")
    print(f"  Average cash advance: {subset['CASH_ADVANCE'].mean():.2f}")
    print(f"  Average credit limit: {subset['CREDIT_LIMIT'].mean():.2f}")
    print(f"  Average payment: {subset['PAYMENTS'].mean():.2f}")
    print(f"  Average tenure: {subset['TENURE'].mean():.1f}months")
    print(f"  Average full payment ratio: {subset['PRC_FULL_PAYMENT'].mean():.2f}")


14. Business Recommendations Summary

Cluster 0: Low-activity customers (Customer count: 2215, Proportion: 24.7%)
  Average balance: 1256.72
  Average purchase amount: 77.68
  Average cash advance: 1029.67
  Average credit limit: 2755.09
  Average payment: 942.12
  Average tenure: 12.0months
  Average full payment ratio: 0.03

Cluster 1: Frequent-purchase customers (Customer count: 1810, Proportion: 20.2%)
  Average balance: 1657.72
  Average purchase amount: 2162.92
  Average cash advance: 280.79
  Average credit limit: 6432.59
  Average payment: 2420.72
  Average tenure: 12.0months
  Average full payment ratio: 0.13

Cluster 2: Cash-advance customers (Customer count: 3597, Proportion: 40.2%)
  Average balance: 468.44
  Average purchase amount: 507.24
  Average cash advance: 31.88
  Average credit limit: 3392.08
  Average payment: 710.77
  Average tenure: 12.0months
  Average full payment ratio: 0.12

Cluster 3: Steady-spending customers (Customer count: 1328, Proportion: 14.8%)
  Av